# Notebook 02 - FAISS: Fast Similarity Search Over Raw Vectors

## What is FAISS?

FAISS (Facebook AI Similarity Search) is a library for efficiently searching large collections
of vectors. It was built by Meta to handle millions or billions of vectors at production scale.

When you have 100 vectors, a brute-force loop over all of them is fine.
When you have 10 million vectors, you need FAISS.

FAISS is not a database. It does not store text, metadata, or anything other than numbers.
It is purely a search engine for vectors. You give it vectors, it gives you nearest neighbors.

## How FAISS Works (Conceptually)

Think of all your vectors as points in a high-dimensional space.
When you give FAISS a query vector, it finds the `k` points that are closest to it.

The simplest FAISS index (`IndexFlatL2`) does this by exact computation - it literally
calculates the distance from the query to every single stored vector and returns the smallest k.
This is called exact search.

For larger collections, FAISS also offers approximate indexes that trade a small amount
of accuracy for much faster search. We will use the flat exact index here for clarity.

## The Distance Metric: L2 vs Cosine

FAISS's flat index uses L2 distance (Euclidean distance) by default.

For vectors that are already normalized to unit length (which `embed()` does),
L2 distance and cosine distance are equivalent - they produce the same ranking.

A lower L2 distance means a higher cosine similarity. Keep that in mind when reading results.

In [ ]:
# Install if needed
# !pip install faiss-cpu sentence-transformers numpy

In [1]:
import sys
sys.path.append('..')

import faiss
import numpy as np
from utils.embedder import embed, embed_batch
from utils.searcher import build_faiss_index, faiss_search

d:\Naveena Natarajan\Tutor\Abhishek-Bangalore-Python\Abhishek_Implementation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1 - Building an Index from a Small Set of Sentences

In [2]:
# Our 'knowledge base' - 8 short sentences on different topics
corpus = [
    "Handle price objections by demonstrating ROI to the prospect.",
    "When a client says it costs too much, reframe around value.",
    "Always ask discovery questions before presenting a solution.",
    "The best salespeople listen 70% of the time.",
    "Our quarterly revenue showed a strong upward trend.",
    "Company income increased 15% compared to last year.",
    "Preheat the oven to 180 degrees before baking the cake.",
    "Add two cups of flour and one teaspoon of baking powder.",
]

# Embed all sentences into a matrix of shape (8, 384)
corpus_embeddings = embed_batch(corpus)
print("Corpus embeddings shape:", corpus_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.78it/s]

Corpus embeddings shape: (8, 384)


In [3]:
# Build the FAISS index
index = build_faiss_index(corpus_embeddings)

print("Index type :", type(index))
print("Vectors stored:", index.ntotal)

Index type : <class 'faiss.swigfaiss_avx2.IndexFlatL2'>
Vectors stored: 8




## Part 2 - Searching the Index

FAISS returns indices (positions in the original corpus list) and distances.
Lower distance = higher similarity.

In [4]:
query = "how should I respond when a prospect says the product is too expensive?"

query_vector = embed(query)
distances, indices = faiss_search(index, query_vector, k=3)

print("Query:", query)
print()
print("Top 3 results:")
for rank, (dist, idx) in enumerate(zip(distances, indices)):
    print(f"  Rank {rank + 1}  |  distance: {dist:.4f}  |  {corpus[idx]}")

Query: how should I respond when a prospect says the product is too expensive?

Top 3 results:
  Rank 1  |  distance: 0.7531  |  When a client says it costs too much, reframe around value.
  Rank 2  |  distance: 0.7853  |  Handle price objections by demonstrating ROI to the prospect.
  Rank 3  |  distance: 1.6134  |  Always ask discovery questions before presenting a solution.


The query uses different words ('too expensive', 'prospect', 'product') than the corpus sentences.
FAISS still finds the right results because it is searching by embedding similarity, not keywords.

## Part 3 - Testing Topic Isolation

In [5]:
# Try a cooking query - should only retrieve the cooking sentences
query_cooking = "what temperature should I set for baking?"
dists, idxs = faiss_search(index, embed(query_cooking), k=3)

print("Query:", query_cooking)
print()
for rank, (d, i) in enumerate(zip(dists, idxs)):
    print(f"  Rank {rank + 1}  |  distance: {d:.4f}  |  {corpus[i]}")

Query: what temperature should I set for baking?

  Rank 1  |  distance: 0.7156  |  Preheat the oven to 180 degrees before baking the cake.
  Rank 2  |  distance: 1.1740  |  Add two cups of flour and one teaspoon of baking powder.
  Rank 3  |  distance: 1.9101  |  Always ask discovery questions before presenting a solution.


In [6]:
# Try a revenue/financial query
query_finance = "how did the company perform financially this year?"
dists, idxs = faiss_search(index, embed(query_finance), k=3)

print("Query:", query_finance)
print()
for rank, (d, i) in enumerate(zip(dists, idxs)):
    print(f"  Rank {rank + 1}  |  distance: {d:.4f}  |  {corpus[i]}")

Query: how did the company perform financially this year?

  Rank 1  |  distance: 0.9622  |  Company income increased 15% compared to last year.
  Rank 2  |  distance: 1.1097  |  Our quarterly revenue showed a strong upward trend.
  Rank 3  |  distance: 1.4385  |  Handle price objections by demonstrating ROI to the prospect.



## Part 4 - Saving and Loading a FAISS Index

You do not want to re-embed your entire corpus every time.
FAISS lets you save the index to disk and reload it later.

In [9]:
# Save the index
faiss.write_index(index, "../data/corpus.faiss")
print("Index saved to ../data/corpus.faiss")

# Load it back
loaded_index = faiss.read_index("../data/corpus.faiss")
print("Loaded index has", loaded_index.ntotal, "vectors")

# Verify the loaded index gives the same results
dists2, idxs2 = faiss_search(loaded_index, embed(query), k=3)
print("Results from loaded index match original:", list(idxs) == list(idxs2))

Index saved to ../data/corpus.faiss
Loaded index has 8 vectors
Results from loaded index match original: False




## Part 6 - FAISS Limitations

FAISS is powerful but it only handles vectors. It does not store:

- The original text of each document
- Metadata (source file, page number, date, author)
- Any ability to filter by metadata before or after search

You have to manage those yourself in a separate Python list or dictionary,
and use the indices FAISS returns to look up the original text.

This works fine for small projects but becomes awkward at scale.
That is why vector databases like ChromaDB exist - they wrap FAISS-style search
with storage, metadata, and filtering built in.

Notebook 03 covers ChromaDB.

## Summary

- FAISS holds vectors in memory and finds nearest neighbors fast.
- `IndexFlatL2` does exact search. Lower distance = more similar meaning.
- For normalized vectors, L2 and cosine distance rank results the same way.
- FAISS stores only numbers - text and metadata are your responsibility.
- You can save the index to disk and reload it to avoid re-embedding.